# Mobile Money Fraud Detection
### Catching fraudulent transactions in a mobile money system

Mobile money — services like MTN MoMo and Airtel Money — is how a huge share of transactions happen across Rwanda and much of Africa, often more than traditional banking. Fraud detection for these systems is a real, high-stakes problem: unlike a bank transfer, mobile money transactions are often instant and hard to reverse.

This notebook builds a fraud detection model on a **realistic synthetic dataset** designed to mirror real mobile money transaction patterns — including how fraud actually tends to look (accounts being rapidly drained via transfers or cash-outs).

**Why synthetic data:** real mobile money transaction data is never publicly released (for obvious privacy and security reasons). This is standard practice in fraud detection research — even well-known research datasets like PaySim (used in published fraud detection papers) are synthetic simulations built to mirror real transaction behavior, not real transaction logs.

**What you'll need:**
```
pip install scikit-learn pandas numpy matplotlib seaborn
```


## Step 1 — Generate a realistic synthetic dataset

We simulate transactions across 5 common mobile money transaction types:
- `CASH_IN` — depositing money into the mobile money account
- `CASH_OUT` — withdrawing money to cash
- `TRANSFER` — sending money to another account
- `PAYMENT` — paying a merchant
- `DEBIT` — a direct debit from the account

**Fraud pattern modeled:** in real mobile money fraud, a common pattern is an account being compromised and then rapidly drained — usually through a `TRANSFER` or `CASH_OUT` that empties most or all of the balance. We build that exact pattern into a small percentage of transactions, while keeping the majority normal — mirroring the heavy class imbalance seen in real fraud detection (fraud is rare, which is itself part of what makes it hard to detect).


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

N_TRANSACTIONS = 20000
FRAUD_RATE = 0.015  # ~1.5% fraud, realistic for this kind of problem

transaction_types = ["CASH_IN", "CASH_OUT", "TRANSFER", "PAYMENT", "DEBIT"]
type_weights = [0.22, 0.28, 0.20, 0.25, 0.05]

rows = []

for i in range(N_TRANSACTIONS):
    is_fraud = np.random.random() < FRAUD_RATE

    if is_fraud:
        # Fraud tends to be TRANSFER or CASH_OUT, draining most of the balance —
        # but not always extreme, so it isn't trivially separable from normal activity
        txn_type = np.random.choice(["TRANSFER", "CASH_OUT"], p=[0.6, 0.4])
        balance_before = np.random.uniform(5000, 500000)
        # Usually drains a lot, but with a long tail down to more moderate amounts
        drain_fraction = np.clip(np.random.normal(0.80, 0.18), 0.15, 1.0)
        amount = balance_before * drain_fraction
        balance_after = balance_before - amount
        odd_hours = [0, 1, 2, 3, 4, 22, 23]
        day_hours = list(range(5, 22))
        odd_weights = [3, 3, 3, 2.5, 2, 2, 1.5]
        day_weights = [1] * len(day_hours)
        all_weights = np.array(odd_weights + day_weights, dtype=float)
        all_weights /= all_weights.sum()
        hour_of_day = np.random.choice(odd_hours + day_hours, p=all_weights)
    else:
        txn_type = np.random.choice(transaction_types, p=type_weights)
        balance_before = np.random.uniform(0, 500000)
        if txn_type in ["CASH_IN"]:
            amount = np.random.uniform(500, 50000)
            balance_after = balance_before + amount
        else:
            # Normal spending is usually small-to-moderate, but occasionally
            # a legitimate large payment/withdrawal drains most of the balance too —
            # this overlap with fraud is what makes the problem realistically hard
            if np.random.random() < 0.08:
                drain_fraction = np.clip(np.random.normal(0.7, 0.15), 0.3, 1.0)
            else:
                drain_fraction = np.clip(np.random.exponential(0.15), 0.01, 0.6)
            amount = balance_before * drain_fraction if balance_before > 0 else np.random.uniform(500, 5000)
            balance_after = max(0, balance_before - amount)
        hour_of_day = np.random.choice(range(24))  # roughly uniform across the day

    rows.append({
        "amount": round(amount, 2),
        "transaction_type": txn_type,
        "balance_before": round(balance_before, 2),
        "balance_after": round(balance_after, 2),
        "hour_of_day": hour_of_day,
        "is_fraud": int(is_fraud),
    })

df = pd.DataFrame(rows)
print(f"Total transactions: {len(df)}")
print(f"Fraud transactions: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.2f}%)")
df.head()


## Step 2 — Feature engineering

In [ ]:
# A very telling signal in real fraud detection: how much of the balance was drained
df["balance_drained_ratio"] = np.where(
    df["balance_before"] > 0,
    (df["balance_before"] - df["balance_after"]) / df["balance_before"],
    0
)

# One-hot encode transaction type
df_encoded = pd.get_dummies(df, columns=["transaction_type"], prefix="type")

print(df_encoded.columns.tolist())
df_encoded.head()


## Step 3 — Explore the fraud patterns visually

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="is_fraud", y="balance_drained_ratio", ax=axes[0])
axes[0].set_title("Balance Drained Ratio: Fraud vs Normal")
axes[0].set_xticklabels(["Normal", "Fraud"])

sns.countplot(data=df, x="hour_of_day", hue="is_fraud", ax=axes[1])
axes[1].set_title("Transaction Hour: Fraud vs Normal")

plt.tight_layout()
plt.show()


**What to look for:** fraud transactions should show a much higher balance-drained ratio (close to 1.0, meaning nearly the whole balance emptied) compared to normal transactions. This single engineered feature will likely turn out to be one of the strongest fraud predictors — a good example of how thoughtful feature engineering often matters more than model complexity.


## Step 4 — Train/test split (with stratification, since fraud is rare)

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df_encoded.columns if c not in ["is_fraud"]]
X = df_encoded[feature_cols]
y = df_encoded["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training fraud rate: {y_train.mean()*100:.2f}%")
print(f"Test fraud rate: {y_test.mean()*100:.2f}%")


## Step 5 — Train the model

Because fraud is rare, a model that just predicts "not fraud" every time would already be ~98.5% accurate — but completely useless. We use `class_weight="balanced"` so the model is penalized more heavily for missing fraud cases, and we'll judge it on **recall** (catching actual fraud) and **precision** (not flagging too many false alarms), not raw accuracy.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
)

model.fit(X_train, y_train)
print("Model trained.")


## Step 6 — Evaluate: precision, recall, and the confusion matrix

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Normal", "Fraud"])
plt.title("Confusion Matrix")
plt.show()


**How to read this for a fraud problem specifically:**
- **Recall** (for the Fraud row) is usually the priority metric — it tells you what percentage of actual fraud your model successfully caught. Missing fraud (a false negative) is typically far more costly than a false alarm.
- **Precision** tells you how many of your fraud *alerts* were actually correct — too low, and you'd be flooding a fraud team with false alarms they'll learn to ignore.
- In a real deployment, you'd tune the decision threshold (currently a default 50%) based on the actual cost of missed fraud vs. the cost of investigating false alarms.


## Step 7 — Which features matter most?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
importances.head(10).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 10 Most Important Features for Fraud Detection")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(importances.head(10))


## Step 8 — Score a new transaction for fraud risk

In [ ]:
def score_transaction(amount, transaction_type, balance_before, balance_after, hour_of_day):
    """Score a single transaction and return a fraud probability."""
    balance_drained_ratio = (
        (balance_before - balance_after) / balance_before if balance_before > 0 else 0
    )

    row = {col: 0 for col in feature_cols}
    row["amount"] = amount
    row["balance_before"] = balance_before
    row["balance_after"] = balance_after
    row["hour_of_day"] = hour_of_day
    row["balance_drained_ratio"] = balance_drained_ratio

    type_col = f"type_{transaction_type}"
    if type_col in row:
        row[type_col] = 1

    input_df = pd.DataFrame([row])[feature_cols]
    fraud_probability = model.predict_proba(input_df)[0][1]

    print(f"Transaction: {transaction_type}, amount={amount}, balance {balance_before} -> {balance_after}")
    print(f"Fraud risk score: {fraud_probability*100:.1f}%")
    return fraud_probability

# Example: a suspicious transaction draining almost the whole balance late at night
score_transaction(
    amount=48000,
    transaction_type="TRANSFER",
    balance_before=50000,
    balance_after=2000,
    hour_of_day=2,
)


In [ ]:
# Example: a normal-looking payment
score_transaction(
    amount=3000,
    transaction_type="PAYMENT",
    balance_before=40000,
    balance_after=37000,
    hour_of_day=14,
)


## Step 9 — Save the model

In [ ]:
import joblib

joblib.dump(model, "mobile_money_fraud_model.joblib")
joblib.dump(feature_cols, "mobile_money_fraud_feature_cols.joblib")
print("Model and feature list saved.")

# To load them again later:
# model = joblib.load("mobile_money_fraud_model.joblib")
# feature_cols = joblib.load("mobile_money_fraud_feature_cols.joblib")


## Next steps

- **Real data**: if you ever get access to real (anonymized) transaction logs — e.g. through a partnership or research access — retrain on real data instead of synthetic. The synthetic patterns here are a reasonable approximation, but real fraud often has messier, evolving patterns.
- **Time-based features**: real fraud detection systems often use velocity features — how many transactions has this account made in the last hour, how much total volume, whether transaction frequency suddenly spiked. These require transaction history per account, which this simplified dataset doesn't model.
- **Real-time scoring**: wrap `score_transaction()` in an API endpoint that could theoretically sit in a real transaction pipeline, scoring each transaction as it happens rather than after the fact.
- **HURO Africa relevance**: if HURO Africa ever processes payments directly, this kind of fraud-scoring layer is directly applicable to protecting transactions on the platform.
- **Explainability**: this project pairs naturally with the explainable credit risk classifier — adding SHAP explanations here would let you explain *why* a specific transaction was flagged, which matters a lot for building trust with users and fraud investigators alike.
